# 2 — Audit semantics and freeze the Stage 2 manifest
This confirms the installed policy/RTC API contract and creates the immutable 360-row design. Stop if any assertion fails.


In [ ]:
import csv, os, subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; PY=Path.home()/"venv-stage1-id/bin/python"; OUT=Path.home()/"stage2"; OUT.mkdir(exist_ok=True)
audit=r'''
import inspect
from lerobot.policies.pi05.configuration_pi05 import PI05Config
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
from async_vla_benchmark.benchmark.rtc import predict_rtc_chunk
config=PI05Config.from_pretrained("lerobot/pi05_libero_finetuned", revision="8e174154ef5f6c60a8da12ae99c303d8963138c1")
config.n_action_steps=25
assert config.n_action_steps == 25
assert config.chunk_size == 50
sig=inspect.signature(PI05Policy.predict_action_chunk)
source=inspect.getsource(predict_rtc_chunk)
for token in ("inference_delay","prev_chunk_left_over","execution_horizon"):
    assert token in source, token
print("PI05 n_action_steps field:", config.n_action_steps)
print("Checkpoint prediction horizon (chunk_size):", config.chunk_size)
print("PI05 predict_action_chunk signature:", sig)
print("PASS: adapter forwards request-specific delay, previous remainder, and execution horizon")
print("DECLARATION: Stage 2 varies policy n_action_steps, RTC execution_horizon, and request threshold together; they remain separately logged.")
'''
env=os.environ.copy(); env["PYTHONPATH"]=str(R); subprocess.run([str(PY),"-c",audit],cwd=R,env=env,check=True)


In [ ]:
MAN=OUT/"stage2_local_sensitivity_manifest.csv"
bench="anonymous-source"
cmd=[str(PY),"-m","async_vla_benchmark.scripts.make_stage2_manifest","--output",str(MAN),"--git-sha",bench,"--lerobot-git-sha","2aba372b4e217cc47db28e0f836859b20d1456c9","--model-revision","8e174154ef5f6c60a8da12ae99c303d8963138c1"]
subprocess.run(cmd,cwd=R,check=True)
GPU=(Path.home()/"stage2_gpu.txt").read_text().strip(); reset_env=os.environ.copy(); reset_env.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg"})
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.resolve_stage2_initializations","--config",str(R/"async_vla_benchmark/configs/stage2.yaml"),"--manifest",str(MAN)],cwd=R,env=reset_env,check=True)
rows=list(csv.DictReader(open(MAN))); assert len(rows)==360
assert sorted({int(r['seed']) for r in rows})==[5,6,7,8,9]
assert sorted({int(r['configured_n_action_steps']) for r in rows})==[10,15,20,25,30,35]
assert sorted({int(r['added_delay_ms']) for r in rows})==[0,100,200,300]
for task in {r['task_key'] for r in rows}:
    for seed in {'5','6','7','8','9'}:
        paired=[r for r in rows if r['task_key']==task and r['seed']==seed]; assert len(paired)==24
        assert len({(r['initialization_index_or_id'],r['initial_state_fingerprint_method'],r['initial_state_fingerprint']) for r in paired})==1
print("PASS: frozen manifest audit 360 rows; 15 matched task×seed reset identities")
print("STOP HERE and review all reset fingerprints before notebook 03.")
